# 3. Olay çalışmaları: bilançolar ve yönetici alımları

## Araştırma soruları
1. **Bilanço tepkisi:** Bilanço açıklamasından sonraki iki gün, sıradan günlerden daha mı oynak?
2. **Bilanço sonrası sürüklenme (PEAD):** İlk tepkinin yönü sonraki iki ayda devam ediyor mu? Akademik literatürde küçük hisselerde görülür; büyük ve çok takip edilen Nasdaq 100 şirketlerinde de var mı?
3. **Yönetici alımları:** Yöneticiler kendi hisselerini piyasadan aldıktan sonra hisse piyasadan iyi gidiyor mu?

## Yöntem
- Olay tarihleri SEC'ten: bilanço için 8-K madde 2.02, yönetici işlemleri için Form 4.
- **Piyasa modeli:** her olay için olaydan 250 ile 30 gün önce arasında `getiri_hisse = α + β · getiri_endeks` tahmin edilir. Olay penceresindeki **anormal getiri** = gerçekleşen − modelin beklediği.
- **CAR** (kümülatif anormal getiri): pencere boyunca anormal getirilerin toplamı. Tepki = gün 0..1, sürüklenme = gün 2..40.
- Testler: tek örneklem t-testi (ortalama ≠ 0?), Welch t-testi (iki grup farkı), Spearman sıra korelasyonu.

In [ ]:
import sys, sqlite3
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from mi import db, stats as st, events as ev, charts
%matplotlib inline
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
con = db.connect()
prices = pd.read_sql('SELECT * FROM prices', con)
close, volume = st.wide(prices), st.wide(prices, 'volume')
r = st.returns(close)
assets = pd.read_sql('SELECT * FROM assets', con).set_index('code')
print(f'{close.shape[1]} varlık, {close.index[0]:%Y-%m-%d} – {close.index[-1]:%Y-%m-%d}, {len(prices):,} fiyat satırı')

In [ ]:
E = pd.read_sql("SELECT code, date FROM filings WHERE items LIKE '%2.02%'", con)
res = ev.earnings_study(r, E)
print(f"{res['n']} bilanço, {res['companies']} şirket, {res['period'][0]} – {res['period'][1]}")

## 3.1 Bilanço günleri ne kadar oynak?

In [ ]:
print(f"Bilanço sonrası 2 günlük ortalama |anormal getiri|: %{res['abs_react_mean']:.2f}")
print(f"Sıradan 2 günlük ortalama |fazla getiri|:         %{res['normal_2d_abs_mean']:.2f}")
print(f"Oran: {res['react_ratio']:.2f} kat")

## 3.2 İlk tepki sürüyor mu?

In [ ]:
charts.event_paths(res['path_days'], res['avg_path'], Path('/tmp/_p.png'))
from IPython.display import Image; display(Image('/tmp/_p.png'))
pd.DataFrame(res['drift_by_bucket']).T[['n', 'mean', 'ci95', 't', 'p', 'hit']]

In [ ]:
d = res['drift_spread']; s = res['spearman']
print(f"Olumlu − olumsuz grup sürüklenme farkı: {d['diff']:.2f} puan, Welch t = {d['t']:.2f}, p = {d['p']:.3f}")
print(f"Tepki ile sonraki getiri arasında Spearman ρ = {s['rho']:.3f}, p = {s['p']:.3f}")
print('Sonuç:', 'Sürüklenme var: piyasa bilançoyu tek seferde fiyatlamıyor.' if d['p'] < .05 and d['diff'] > 0 else 'Sürüklenme yok: büyük şirketlerde bilgi hızla fiyatlanıyor (verimli piyasa ile uyumlu).')

## 3.3 En sert bilanço tepkileri

In [ ]:
pd.DataFrame(res['biggest'])

## 3.4 Yönetici alımları ve plansız satışlar

In [ ]:
T = pd.read_sql('SELECT * FROM insider_trades', con)
ins = ev.insider_study(r, T)
print({k: f"{d['raw_events']} bildirim → {d['car40'].get('n', 0)} olay, {d['companies']} şirket, dönem {d['period']}" for k, d in ins.items()})
pd.DataFrame({(k, w): d[w] for k, d in ins.items() for w in ('car20', 'car40')}).T

## Sınırlamalar
- **Örneklem:** Yönetici alımları Nasdaq 100'de nadir; sonuçların güven aralığı geniş. Satışların çoğu önceden planlanmış (10b5-1) olduğu için yalnızca plansız satışlar incelendi.
- **Hayatta kalma yanlılığı:** Evren bugünkü Nasdaq 100 listesi; geçmişte endeksten çıkan şirketler dahil değil, bu da geçmiş getirileri olduğundan iyi gösterebilir.
- **Zamanlama:** 8-K'nın gün içi saati bilinmediği için tepki penceresi iki gün (0–1) alındı.
- **Beklenti verisi yok:** Bilanço "sürprizi" analist beklentisiyle değil, fiyat tepkisiyle ölçüldü.